# 1 -  Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
import warnings
warnings.filterwarnings("ignore")

# 2 - Load the data and Preprocessing

In [ ]:
df=pd.read_csv('Assessment Data2.csv')

creating timestamps for the data

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'],format='mixed', errors='coerce')


In [ ]:
df.dtypes

In [ ]:
df = df.sort_values(by='Order_Date')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df = df.dropna(subset=['Order_Date'])


In [ ]:
df.tail()

creating line items based on instructions

In [ ]:
def create_line_item_id(row):
  return f"{row['Order_ID']}_{row['Product_ID']}"


In [ ]:
df['Line_Item_ID'] = df.apply(create_line_item_id, axis=1)

In [ ]:
df.shape

creating aggregation logic for handling other columns

In [ ]:
grouped_df = df.groupby('Line_Item_ID').agg({
    'Order_ID': 'first',          
    'Order_Date': 'max',          
    'Product_ID': 'first',        
    'Quantity': 'sum'
}).reset_index()

In [ ]:
grouped_df.shape

visualizing the data set

In [ ]:
grouped_df.head()

In [ ]:
grouped_df = grouped_df.sort_values(by='Order_Date')

In [ ]:
grouped_df.head()

In [ ]:
new_df = grouped_df.reset_index()

In [ ]:
new_df.dtypes

In [ ]:
new_df.drop(columns='index',inplace=True)
new_df.rename(columns={'level_0':'Index'},inplace=True)


In [ ]:
new_df.head()

In [ ]:
new_df.tail()

In [ ]:
new_df['Product_ID'].value_counts()

there are 63244 different items sold

In [ ]:
new_df.head()

Checkin for outliers:

In [ ]:
new_df['Quantity'].describe()

In [ ]:
new_df['Quantity'].plot(kind='box')
plt.show()

Using IQR, we can see that std is very large, and altho mean and median are relatively close, max is 8000. so this suggest the occurence of outliers which could skew the data and we can confirm this using the box plot. 

In [ ]:
from scipy import stats
outliers = new_df[np.abs(stats.zscore(new_df['Quantity'])) >= 3]

In [ ]:
outliers

As observed, our dataset contains **outliers** which represent large orders. Due to the valuable insights these data points provide, we must handle them carefully. Dropping these observations is not an option, so we will explore alternative methods:

**Transformation:** 
    
Applying a mathematical function, such as logarithmic transformation, to the 'quantity' column can help reduce the impact of outliers. This approach allows us to maintain the outliers in the dataset while mitigating their influence on the overall analysis.
    
**Outlier encoding:**

Creating a new column to flag outliers allows us to treat them as distinct entities during analysis. This method enables us to retain outliers and leverage the information they provide, which can be beneficial in certain machine learning algorithms or statistical models.

In [ ]:
new_df['outlier_flag'] = (np.abs(stats.zscore(new_df['Quantity']))>= 3).astype(int)

In [ ]:
new_df[new_df['outlier_flag']==1]

In [ ]:
data=new_df.drop(columns=['Line_Item_ID','Order_ID'])

**Our objective is to project three key values:** 'item\_id', 'quantity', and the outlier flag, indicating whether an order is an outlier or not. For this purpose, we will utilize a multi-output regression model, which is suitable for handling multiple target variables.
To enhance the performance and efficiency of our regression model, we will employ label encoding to convert the categorical 'item\_id' into numerical values. Label encoding facilitates easier computation and improves the model's overall speed. We have utilized the LabelEncoder module to achieve this transformation.

With the categorical data encoded numerically and the target variables defined, we are now equipped to proceed with the multi-output regression analysis.

In [ ]:
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

In [ ]:
le = LabelEncoder()
data['Product_ID_encoded'] = le.fit_transform(data['Product_ID'])

In [ ]:
data.set_index('Order_Date',inplace=True)
data.head()

In [ ]:
data_encoding=data[['Product_ID','Product_ID_encoded']].reset_index()
data_encoding.drop(columns='Order_Date',inplace=True)

In [ ]:
data.drop(columns='Product_ID',inplace=True)
data.head()

In [ ]:
data['Product_ID_encoded'].value_counts()

we can see that each 63244 product_ids are now mapped to a unique integer 

In [ ]:
data.dtypes

# 3 - Feature engineering - feature extraction

In [ ]:
def create_features(df):
    """
    Create time series features based on time series index.
    """
    df['day'] = df.index.day
    df['month'] = df.index.month
    df ['quarter'] = df.index.quarter
    df['year'] = df.index.year
    
    return df

data = create_features(data)

In [ ]:
data.head()

# 4 - Regression Model

creating Training and Target arrays to feed to the model

In [ ]:
X=np.array((data['day'],data['month'],data['quarter'],data['year'])).T

In [ ]:
X.shape

In [ ]:
y=np.array((data['Product_ID_encoded'],data['Quantity'],data['outlier_flag'])).T

In [ ]:
y.shape

### 4 - 1 - Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Define the test size, e.g., 20% of the data
test_size = 0.2

# Split the X and y data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)


creating model instance and fitting

The Random Forest Regressor is a learning method that operates by constructing multiple decision trees at training time and outputting the mean of the classes (classification) or mean prediction of the individual trees (regression). This technique helps reduce overfitting and improve generalization capabilities, as it combines predictions from multiple models and can capture non-linear relationships between features and targets.

To build the model, I have initialized the RandomForestRegressor class and fitted the model. This trains the Random Forest Regressor model on your multi-output regression task, enabling it to learn the underlying patterns and relationships within the data.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
# Define parameters
#n_estimators = 350          # Increase the number of trees in the forest
#max_depth = 10            # Let the model decide the depth of the trees
#min_samples_split = 10      # Increase the minimum number of samples required to split an internal node
#min_samples_leaf = 2        # Increase the minimum number of samples required to be at a leaf node


# Initialize RandomForestRegressor with the defined parameters
rdregressor = RandomForestRegressor()

In [ ]:
rdregressor.fit(X_train,y_train)


sample test data

In [ ]:
test_data=[[ 3, 4,  1, 2001]]

make prediction on sample tes data

In [ ]:
rdregressor.predict(test_data).astype(int)

Now that I trained the model, I will check the performance, lets import some metrics:

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Predict the target values for the test set
y_pred = rdregressor.predict(X_test)

for i in range(3):
    # Compute and print the evaluation metrics for each target variable
    print("Target variable", i+1)
    print("Mean Squared Error (MSE):", mean_squared_error(y_test[:, i], y_pred[:, i]))
    print("Mean Absolute Error (MAE):", mean_absolute_error(y_test[:, i], y_pred[:, i]))
    print("R-squared (R?):", r2_score(y_test[:, i], y_pred[:, i]))
    print()

Now that we've run the model on the test set and generated performance metrics, a few important points should be noted. Fine-tuning the model can be time-consuming, as it requires running the model multiple times with different hyperparameters to find the best possible outcome. Additionally, since we are projecting three variables based on only four features (which may not be entirely independent), the model might exhibit some weaknesses. In this case, it would be beneficial to utilize more advanced feature engineering techniques, such as grouping products, regions, and zones, to extract more meaningful information from the data.

Furthermore, it is essential to consider the trade-off between model complexity and performance. A more complex model might overfit the training data, resulting in poor generalization to unseen data. Regularization techniques, cross-validation, and pruning can help alleviate overfitting issues and improve model performance.

Lastly, exploring alternative models or ensemble methods could potentially yield better results. This could involve experimenting with various regression models, or utilizing techniques such as bootstrapping or bagging to combine the strengths of multiple models. The ultimate goal is to find the best model that can accurately predict the target variables without sacrificing generalization ability.


Defining a function to create a (near) empty data frame which will be filled with next years projcetions

In [ ]:
def create_projection_df(year):
  days = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31")
  df = pd.DataFrame({'day': days.day,
                     'month': days.month,
                     'quarter': days.quarter,
                     'year': days.year})
  return df



In [ ]:
# Create the projection DataFrame
projection_df = create_projection_df(2004)

In [ ]:
projection_df.head()

Make prediction for the year 2004

In [ ]:
# Make predictions
predictions = rdregressor.predict(projection_df[['day', 'month', 'quarter', 'year']]).astype(int)


reverting back the encoded product_id, and filling the dataframe

In [ ]:
# Add predictions to the projection DataFrame
projection_df['Predicted_Quantity'] = predictions[:, 1].astype(int)
projection_df['Predicted_Product_ID'] = le.inverse_transform(predictions[:, 0].astype(int))
projection_df['Projected_outlier_flag'] = predictions[:, 2].astype(int)

In [ ]:
projection_df.head()